# Lab03 — Análisis de cointegración
**Isabela Torres-Septien Uribe · Andrea Santoyo Vega · Abdon Islas León**

Tablas de análisis. Todo lee de `artifacts/`.(Las figuras están en `Figuras_Lab03.ipynb`.)

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 60)

REPO_ROOT = os.path.abspath(".")   # ajusta si el notebook NO esta en la raiz del repo
os.chdir(REPO_ROOT); sys.path.insert(0, REPO_ROOT)
ART = "artifacts"
chosen = json.load(open(f"{ART}/chosen_group.json"))
cols = chosen["grupo"].split("+")
print("Repo:", REPO_ROOT, "| grupo elegido:", chosen["grupo"], "| vector:", chosen["vector"])

Repo: c:\Users\Isabela\Documents\Iteso\9no Semestre\Trading\Lab03_MyST_Isabela-Andrea-Abdon-repository | grupo elegido: AMT+SBAC | vector: [1.0, -0.8631]


### Orden de integración
Orden de cada activo (ADF en nivel, ADF en diferencia, KPSS); confirma que las series sean I(1) antes de cointegrar.

In [2]:
pd.read_csv(f"{ART}/integration_order.csv")

,ticker,ADF_nivel_p,ADF_diff_p,KPSS_nivel_p,orden
0,AMT,0.7223,0.0,0.01,I(1)
1,CCI,0.8722,0.0,0.01,I(1)
2,SBAC,0.8905,0.0,0.01,I(1)
3,VZ,0.1884,0.0,0.01,I(1)
4,T,0.0161,0.0,0.01,revisar
5,TMUS,0.7938,0.0,0.01,I(1)


### Pre-filtro Engle-Granger (15 pares)
Correlación móvil mediana y EG en ambos sentidos por par; es una heurística, el criterio definitivo es Johansen.

In [3]:
pd.read_csv(f"{ART}/engle_granger_pairs.csv")

,par,corr_movil_mediana,pasa_corr,EG_p_AB,EG_p_BA,pasa_EG
0,AMT-CCI,0.7623,True,0.6347,0.6200,False
1,AMT-SBAC,0.7532,True,0.0676,0.1052,False
2,AMT-VZ,0.2518,False,0.1044,0.0401,True
3,AMT-T,0.1561,False,0.8730,0.0641,False
4,AMT-TMUS,0.1747,False,0.2984,0.3198,False
5,CCI-SBAC,0.7419,True,0.0520,0.0664,False
6,CCI-VZ,0.2288,False,0.3819,0.1073,False
7,CCI-T,0.1816,False,0.9328,0.0346,True
8,CCI-TMUS,0.1939,False,0.0770,0.1025,False
9,SBAC-VZ,0.1879,False,0.3459,0.0715,False


### Falsos positivos esperados y etapa de caída
FP esperados = n_pruebas × 0.05. Abajo, cuántos de los 57 grupos cayeron en cada etapa del embudo.

In [4]:
display(pd.read_csv(f"{ART}/false_positives.csv"))
res = pd.read_csv(f"{ART}/screening_all_groups.csv")
display(res["etapa_caida"].value_counts().to_frame("conteo"))

,prueba,n_tests,alpha,FP_esperados
0,Engle-Granger,30,0.05,1.50
1,Johansen,57,0.05,2.85


,conteo
etapa_caida,
johansen,54
PASA,3


### Grupos probados (embudo completo)
Primero los que pasan el embudo; abajo la tabla de **todos los grupos probados** con su etapa de descarte (lo exige el lab).

In [5]:
res = pd.read_csv(f"{ART}/screening_all_groups.csv")
print("Grupos que pasan el embudo completo:")
display(res[res["etapa_caida"] == "PASA"].sort_values("ADF_spread_p"))
print(f"Todos los grupos probados ({len(res)}):")
display(res)

Grupos que pasan el embudo completo:


,grupo,n,corr_min_par,rank_traza,rank_maxeig,bandera_roja,activo_no_I1,ADF_spread_p,vida_media,vector,pata_normalizacion,etapa_caida
0,AMT+SBAC+T,3,0.1369,2,1,True,['T'],0.0041,33.3,"[1.0, -0.8585, -0.4294]",AMT,PASA
1,AMT+SBAC,2,0.7532,1,1,False,[],0.0120,41.3,"[1.0, -0.8631]",AMT,PASA
2,VZ+T,2,0.6485,1,1,True,['T'],0.0158,48.9,"[0.0555, 1.0]",T,PASA


Todos los grupos probados (57):


,grupo,n,corr_min_par,rank_traza,rank_maxeig,bandera_roja,activo_no_I1,ADF_spread_p,vida_media,vector,pata_normalizacion,etapa_caida
0,AMT+SBAC+T,3,0.1369,2,1,True,['T'],0.0041,33.3,"[1.0, -0.8585, -0.4294]",AMT,PASA
1,AMT+SBAC,2,0.7532,1,1,False,[],0.0120,41.3,"[1.0, -0.8631]",AMT,PASA
2,VZ+T,2,0.6485,1,1,True,['T'],0.0158,48.9,"[0.0555, 1.0]",T,PASA
3,AMT+SBAC+VZ+T,4,0.1369,1,0,True,['T'],0.0000,23.4,"[1.0, -0.7131, -0.5189, -0.3515]",AMT,johansen
4,AMT+CCI+SBAC+VZ+T,5,0.1369,0,0,True,['T'],0.0000,21.9,"[1.0, 0.2324, -0.8948, -0.5133, -0.2984]",AMT,johansen
5,AMT+SBAC+VZ+T+TMUS,5,0.1369,0,0,True,['T'],0.0000,23.3,"[1.0, -0.7238, -0.5148, -0.3581, 0.0111]",AMT,johansen
6,AMT+CCI+SBAC+VZ+TMUS,5,0.1747,0,0,False,[],0.0001,22.9,"[-0.8552, -0.6454, 1.0, 0.4091, 0.1519]",SBAC,johansen
7,AMT+CCI+SBAC+VZ+T+TMUS,6,0.1369,0,0,True,['T'],0.0001,21.2,"[-0.9253, -0.5516, 1.0, 0.4393, 0.1713, 0.1226]",SBAC,johansen
8,AMT+CCI+SBAC+VZ,4,0.1879,0,0,False,[],0.0003,25.8,"[-0.9785, -0.3805, 1.0, 0.5008]",SBAC,johansen
9,AMT+CCI+VZ+T,4,0.1561,0,0,True,['T'],0.0003,34.9,"[-0.9965, 0.731, 0.9473, 1.0]",T,johansen


### VECM — vector β y velocidad de ajuste α
β es el vector de cointegración; α dice qué activo corrige el desequilibrio (el de |α| mayor y significativo).

In [6]:
print("Vector de cointegracion (beta):"); display(pd.read_csv(f"{ART}/vecm_beta.csv"))
print("Velocidad de ajuste (alpha):");        display(pd.read_csv(f"{ART}/vecm_alpha.csv"))

Vector de cointegracion (beta):


,ticker,beta
0,AMT,1.0000
1,SBAC,-0.8631


Velocidad de ajuste (alpha):


,ticker,alpha,pval
0,AMT,-0.0017,0.7926
1,SBAC,0.0163,0.0221


### Robustez fuera de muestra
Vector estimado solo en sub-train y aplicado a sub-test/sub-val; si el spread deja de ser estacionario, la relación no es estructural.

In [11]:
!python -m cointegration.robustez_oos

Grupo: AMT+SBAC  |  vector (sub-train, pata=AMT): [1.0, -0.9086]
sub_muestra  n_obs  ADF_spread_p  estacionario  vida_media
  sub_train    995        0.0136          True        42.8
   sub_test    332        0.6147         False        20.6
    sub_val    332        0.8497         False        98.8

>>> SE ROMPE en sub-test: el spread deja de ser estacionario fuera de muestra.
    La relacion de equilibrio NO es estable -> posible artefacto/sobreajuste.


In [12]:
pd.read_csv("artifacts/robustez_oos.csv")

,sub_muestra,n_obs,ADF_spread_p,estacionario,vida_media
0,sub_train,995,0.0136,True,42.8
1,sub_test,332,0.6147,False,20.6
2,sub_val,332,0.8497,False,98.8
